# 02 — Data Preprocessing 

This notebook performs the complete preprocessing pipeline using the project's
predefined utility functions:

- `prepare_features_and_labels()` for feature extraction and label mapping  
- `BreastCancerPreprocessor` for MinMax scaling and optional L2 normalization  

This ensures consistency across the entire project, avoids duplicated code, and 
produces reproducible processed datasets ready for classical and quantum SVM models.


In [19]:
import os
import sys
import numpy as np
import pandas as pd

# Add project root so that src/ works
sys.path.append(os.path.abspath(".."))

from src.data.preprocessing import (
    load_raw_data,
    prepare_features_and_labels,
    BreastCancerPreprocessor,
    get_train_test_split,
)

print("Imports OK")


Imports OK


In [20]:
df = load_raw_data("../data/raw/data.csv")

X, y = prepare_features_and_labels(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("NaN in X?", np.isnan(X).any())
print("Columns after feature extraction:", X.shape[1])


X shape: (569, 30)
y shape: (569,)
NaN in X? False
Columns after feature extraction: 30


In [21]:
pre = BreastCancerPreprocessor(apply_l2=False)

X_processed = pre.fit_transform(X)

print("NaN in X_processed?", np.isnan(X_processed).any())
print("Min:", X_processed.min(), "Max:", X_processed.max())


NaN in X_processed? False
Min: 0.0 Max: 1.0000000000000002


In [22]:
X_train, X_test, y_train, y_test = get_train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=True,
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test  shape:", X_test.shape, y_test.shape)


Train shape: (455, 30) (455,)
Test  shape: (114, 30) (114,)


In [23]:
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save csv
processed_df = pd.DataFrame(X_processed)
processed_df["diagnosis"] = y
processed_df.to_csv(os.path.join(OUTPUT_DIR, "processed_data.csv"), index=False)

# Save numpy arrays
np.save(os.path.join(OUTPUT_DIR, "X.npy"), X_processed)
np.save(os.path.join(OUTPUT_DIR, "y.npy"), y)

np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

print("Files saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))


Files saved to: ../data/processed
['processed_data.csv', 'X.npy', 'X_test.npy', 'X_test_qsvm.npy', 'X_train.npy', 'X_train_qsvm.npy', 'y.npy', 'y_test.npy', 'y_train.npy']


In [24]:
X_saved = np.load("../data/processed/X.npy")
X_train_saved = np.load("../data/processed/X_train.npy")

print("X_saved equals X_processed?  ", np.allclose(X_saved, X_processed))
print("X_train matches?             ", np.allclose(X_train_saved, X_train))

print("\nMax diff X:", np.nanmax(np.abs(X_saved - X_processed)))
print("Max diff Train:", np.nanmax(np.abs(X_train_saved - X_train)))


X_saved equals X_processed?   True
X_train matches?              True

Max diff X: 0.0
Max diff Train: 0.0


In [25]:
pre_qsvm = BreastCancerPreprocessor(apply_l2=True)

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train_qsvm = pre_qsvm.fit_transform(X_train)
X_test_qsvm  = pre_qsvm.transform(X_test)

print("X_train_qsvm shape:", X_train_qsvm.shape)
print("X_test_qsvm shape:", X_test_qsvm.shape)

np.save(os.path.join(OUTPUT_DIR, "X_train_qsvm.npy"), X_train_qsvm)
np.save(os.path.join(OUTPUT_DIR, "X_test_qsvm.npy"), X_test_qsvm)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

print("Saved to:", OUTPUT_DIR)


X_train_qsvm shape: (455, 30)
X_test_qsvm shape: (114, 30)
Saved to: ../data/processed


In [26]:
import numpy as np

row_norms = np.linalg.norm(X_train_qsvm, axis=1)
print("Avg norm:", row_norms.mean())
print("Min norm:", row_norms.min())
print("Max norm:", row_norms.max())


Avg norm: 1.0
Min norm: 0.9999999999999997
Max norm: 1.0000000000000002


## Preprocessing Summary

- Successfully extracted 30 numeric features.
- Removed NaN-only columns automatically.
- Applied MinMax scaling (no L2).
- Achieved clean, NaN-free processed datasets.
- Saved:
  - processed_data.csv
  - X.npy, y.npy
  - X_train.npy, X_test.npy
  - y_train.npy, y_test.npy

This dataset is now ready for use in the classical SVM baseline.


In [27]:
"""import os

folder = "../data/processed"

for fname in os.listdir(folder):
    path = os.path.join(folder, fname)
    if os.path.isfile(path):
        os.remove(path)

print("Processed folder cleaned.")"""


'import os\n\nfolder = "../data/processed"\n\nfor fname in os.listdir(folder):\n    path = os.path.join(folder, fname)\n    if os.path.isfile(path):\n        os.remove(path)\n\nprint("Processed folder cleaned.")'